# President Tweets on Risk Metrics

James 

# Imports

In [89]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import warnings
warnings.filterwarnings('ignore')

# Set plotting style if needed
import matplotlib.pyplot as plt
plt.style.use('ggplot')

## Load data

### Financial Data

In [90]:
snp = pd.read_csv('data/snp.csv')
snp

,Date,Close,High,Low,Open,Volume
0,2007-01-03,1416.599976,1429.420044,1407.859985,1418.030029,3429160000
1,2007-01-04,1418.339966,1421.839966,1408.430054,1416.599976,3004460000
2,2007-01-05,1409.709961,1418.339966,1405.750000,1418.339966,2919400000
3,2007-01-08,1412.839966,1414.979980,1403.969971,1409.260010,2763340000
4,2007-01-09,1412.109985,1415.609985,1405.420044,1412.839966,3038380000
...,...,...,...,...,...,...
4023,2022-12-23,3844.820068,3845.800049,3797.010010,3815.110107,2819280000
4024,2022-12-27,3829.250000,3846.649902,3813.219971,3843.340088,3030300000
4025,2022-12-28,3783.219971,3848.320068,3780.780029,3829.560059,3083520000
4026,2022-12-29,3849.280029,3858.189941,3805.449951,3805.449951,3003680000


In [91]:
# Load financial datasets
vix = pd.read_csv('data/VIX_DAILY.csv', na_values='.')
treasury = pd.read_csv('data/3mo_treasury.csv', na_values='.')
oil = pd.read_csv('data/oil_px.csv')
snp = pd.read_csv('data/snp.csv')

# Standardize date columns to datetime
vix['date'] = pd.to_datetime(vix['observation_date'])
treasury['date'] = pd.to_datetime(treasury['observation_date'])
oil['date'] = pd.to_datetime(oil['observation_date'])
snp['date'] = pd.to_datetime(snp['Date'])

# Clean and convert values to numeric
vix['VIXCLS'] = pd.to_numeric(vix['VIXCLS'], errors='coerce')
treasury['DGS3MO'] = pd.to_numeric(treasury['DGS3MO'], errors='coerce')
oil['DCOILWTICO'] = pd.to_numeric(oil['DCOILWTICO'], errors='coerce')
snp['value'] = pd.to_numeric(snp['Close'], errors='coerce')

# Merge financial data on date
risk_df = vix[['date', 'VIXCLS']].merge(treasury[['date', 'DGS3MO']].rename(columns={'DGS3MO': 'Treasury_Yield'}), on='date', how='outer')
risk_df = risk_df.merge(oil[['date', 'DCOILWTICO']].rename(columns={'DCOILWTICO': 'Oil_Price'}), on='date', how='outer')
risk_df = risk_df.merge(snp[['date', 'value']].rename(columns={'value': 'SNP500'}), on='date', how='outer')


# Sort and calculate daily changes (Risk metric is often the daily change or % change)
risk_df = risk_df.sort_values('date').set_index('date')
risk_df['VIX_Change'] = risk_df['VIXCLS'].diff() # Daily absolute change in VIX
risk_df['Treasury_Return'] = risk_df['Treasury_Yield'].diff()
risk_df['Oil_Return'] = risk_df['Oil_Price'].pct_change() # Daily return in Oil
risk_df['SNP500_Return'] = risk_df['SNP500'].pct_change() # Daily return in equities
risk_df = risk_df.dropna()

In [92]:
risk_df[::-1]

,VIXCLS,Treasury_Yield,Oil_Price,SNP500,VIX_Change,Treasury_Return,Oil_Return,SNP500_Return
date,,,,,,,,
2022-12-30,21.67,4.42,80.16,3839.500000,0.23,-0.03,0.022058,-0.002541
2022-12-29,21.44,4.45,78.43,3849.280029,-0.70,-0.01,-0.005831,0.017461
2022-12-28,22.14,4.46,78.89,3783.219971,0.49,0.00,-0.007048,-0.012021
2022-12-23,20.87,4.34,79.57,3844.820068,-1.10,-0.01,0.024331,0.005868
2022-12-22,21.97,4.35,77.68,3822.389893,1.90,0.02,-0.006268,-0.014452
...,...,...,...,...,...,...,...,...
2007-01-10,11.47,5.09,53.95,1414.849976,-0.44,0.01,-0.030548,0.001940
2007-01-09,11.91,5.08,55.65,1412.109985,-0.09,0.00,-0.007668,-0.000517
2007-01-08,12.00,5.08,56.08,1412.839966,-0.14,0.03,-0.003731,0.002220


### Twitter Data

In [93]:
# Load Twitter datasets
biden = pd.read_csv('data/JoeBiden.csv')
obama = pd.read_csv('data/obama.csv')
trump = pd.read_csv('data/rollcall_social_posts.csv')


trump = trump[trump['content'] != '[Video]']
trump = trump[trump['date'].apply(lambda x: x.startswith("20"))] # ensure proper date formatting
trump['text'] = trump['content']

# Standardize text and date columns
biden = biden[['date', 'content']].rename(columns={'content': 'text'})
obama = obama[['Timestamp', 'Text']].rename(columns={'Timestamp': 'date', 'Text': 'text'})
trump = trump[['date', 'text', 'platform']]

biden['platform'] = 'Twitter'
obama['platform'] = 'Twitter'

# Add author identifiers
biden['author'] = 'Biden'
obama['author'] = 'Obama'
trump['author'] = 'Trump'

# Combine all tweets
tweets = pd.concat([biden, obama, trump], ignore_index=True)

# 1. Convert to datetime with utc=True to handle mixed formats/zones
tweets['date'] = pd.to_datetime(tweets['date'], format='mixed', utc=True)

# 2. Strip the timezone info (making them 'naive') so they can be easily compared/grouped
tweets['date'] = tweets['date'].dt.tz_localize(None)

# 3. Now extract the date and convert back to datetime for grouping
tweets['date'] = pd.to_datetime(tweets['date'].dt.date)

# 4. Drop any that failed to parse
tweets = tweets.dropna(subset=['date', 'text'])

# Define presidency dates to create the 'is_president' flag
def check_if_president(row):
    d = row['date']
    author = row['author']
    if author == 'Obama' and ('2009-01-20' <= str(d.date()) < '2017-01-20'):
        return 1
    elif author == 'Trump' and ('2017-01-20' <= str(d.date()) < '2021-01-20'):
        return 1
    elif author == 'Biden' and ('2021-01-20' <= str(d.date())):
        return 1
    return 0

tweets['is_president'] = tweets.apply(check_if_president, axis=1)
# obama['date'] = pd.to_datetime(obama['date'], errors='coerce')
# trump['date'] = pd.to_datetime(trump['date'], errors='coerce')

# Group tweets by day (concatenate text, take max of is_president)
daily_tweets = tweets.groupby(['date']).agg({
    'text': lambda x: ' '.join(x.astype(str)),
    'is_president': 'max',
    'author': lambda x: ', '.join(x.unique()), # Lists unique authors for that day
    'platform': lambda x: ', '.join(x.unique()) # Lists unique sites posted on for that day
}).reset_index()

### Merge dfs

In [94]:
# Merge the daily tweets with the daily risk metrics
df_merged = risk_df.merge(daily_tweets.set_index('date'), left_index=True, right_index=True, how='inner')

# Clean the combined text: lowercasing, removing URLs, special characters
def clean_text(text):
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove punctuation/numbers
    return text.lower()

df_merged['clean_text'] = df_merged['text'].apply(clean_text)

## NLP Text extraction

In [95]:
# Using TF-IDF to extract top 100 most frequent/important keywords to prevent overfitting
vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df_merged['clean_text'])

# Create a DataFrame for the TF-IDF features
keyword_features = pd.DataFrame(
    tfidf_matrix.toarray(), 
    columns=vectorizer.get_feature_names_out(),
    index=df_merged.index
)

# Join the keyword features back to our main dataset
df_final = pd.concat([df_merged, keyword_features], axis=1)
df_final = df_final.dropna()

In [96]:
df_final[df_final['is_president'] == 1]

,VIXCLS,Treasury_Yield,Oil_Price,SNP500,VIX_Change,Treasury_Return,Oil_Return,SNP500_Return,text,is_president,...,want,watch,way,whitehouse,win,work,working,world,year,years
date,,,,,,,,,,,,,,,,,,,,,
2009-03-25,42.25,0.19,52.24,813.880005,-0.68,-0.02,-0.020990,0.009626,"Barack Obama\n@BarackObama\n·\nMar 25, 2009",1,...,0.000000,0.00000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000
2009-04-28,37.95,0.13,49.01,855.159973,-0.37,0.00,-0.005681,-0.002741,"Barack Obama\n@BarackObama\n·\nApr 28, 2009",1,...,0.000000,0.00000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000
2009-04-30,36.50,0.14,50.35,872.809998,0.42,0.03,0.003188,-0.000950,"Barack Obama\n@BarackObama\n·\nApr 30, 2009",1,...,0.000000,0.00000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000
2009-05-01,35.30,0.16,52.18,877.520020,-1.20,0.02,0.036346,0.005396,"Barack Obama\n@BarackObama\n·\nMay 1, 2009",1,...,0.000000,0.00000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000
2009-05-12,31.80,0.19,58.81,908.349976,-1.07,0.00,0.017650,-0.000979,"Barack Obama\n@BarackObama\n·\nMay 12, 2009 Mi...",1,...,0.000000,0.00000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-12,25.00,4.38,72.96,3990.560059,2.17,0.07,0.026882,0.014279,The Respect for Marriage Act will safeguard th...,1,...,0.000000,0.00000,0.000000,0.0,0.000000,0.0,0.056642,0.0,0.106098,0.000000
2022-12-13,22.55,4.35,75.44,4019.649902,-2.45,-0.03,0.033991,0.007290,"Today, I signed the Respect for Marriage Act i...",1,...,0.000000,0.00000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000
2022-12-14,21.14,4.33,77.14,3995.320068,-1.41,-0.02,0.022534,-0.006053,"Starting January 1, seniors with diabetes on M...",1,...,0.000000,0.14051,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000


### regression

In [97]:
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer
# Fit Ordinary Least Squares (OLS) Regression
# model = sm.OLS(Y, X).fit()
model1 = smf.ols(formula="VIX_Change ~ is_president + C(author) + C(platform)", data=df_final).fit()
model2 = smf.ols(formula="Treasury_Return ~ is_president + C(author) + C(platform)", data=df_final).fit()
model3 = smf.ols(formula="Oil_Return ~ is_president + C(author) + C(platform)", data=df_final).fit()
model4 = smf.ols(formula="SNP500_Return ~ is_president + C(author) + C(platform)", data=df_final).fit()

# Print the regression summary
# print(model.summary2())


sg = Stargazer([model1, model2, model3, model4])
sg.title("OLS with Presidential Control Results")
sg.custom_columns(
    ["VIX Change", "Treasury Return", "Oil Return", "S&P500 Return"],
    [1, 1, 1, 1]
)
with open("outfile.html", "w") as f:
    f.write(sg.render_html())


## Final words 

In [98]:
res_words = {}
fin_vars = ['VIX_Change','Treasury_Return', 'Oil_Return','SNP500_Return']
for fin_var in fin_vars:
    # regression for keywords
    Y = df_final[fin_var] 

    # Predictors: Keyword frequencies + the "is_president" effect + a constant intercept
    X = df_final[['is_president'] + list(keyword_features.columns)]
    X = sm.add_constant(X)

    # Fit Ordinary Least Squares (OLS) Regression
    model = sm.OLS(Y, X).fit()

    # Extract P-values and Coefficients from the model
    results_df = pd.DataFrame({
        'Coefficient': model.params,
        'P-Value': model.pvalues
    })

    # Filter out the constant to focus purely on features
    results_df = results_df.drop('const', errors='ignore')

    # Determine significance (p < 0.05) and sort by absolute coefficient impact
    significant_factors = results_df[results_df['P-Value'] < 0.1]
    
    res_words[fin_var] = significant_factors.reset_index()['index']
    significant_factors['Abs_Impact'] = significant_factors['Coefficient'].abs()
    significant_factors = significant_factors.sort_values(by='Abs_Impact', ascending=False)

    print("Top Impactful Words / Features (Statistically Significant):")
    display(significant_factors.head(20))

    # Specifically check the effect of being in office
    print("\nEffect of being President on Risk:")
    if 'is_president' in significant_factors.index:
        effect = significant_factors.loc['is_president', 'Coefficient']
        print(f"Statistically significant effect. Coefficient: {effect:.4f}")
    else:
        print(f"Not statistically significant at p < 0.05. Coefficient was {results_df.loc['is_president', 'Coefficient']:.4f} (p-value: {results_df.loc['is_president', 'P-Value']:.4f})")

Top Impactful Words / Features (Statistically Significant):


,Coefficient,P-Value,Abs_Impact
care,1.766155,0.000662,1.766155
strong,1.533791,0.022176,1.533791
did,-1.439497,0.026865,1.439497
know,-1.168448,0.079922,1.168448
china,1.098564,0.024482,1.098564
democrats,0.988315,0.051080,0.988315



Effect of being President on Risk:
Not statistically significant at p < 0.05. Coefficient was -0.0895 (p-value: 0.4252)
Top Impactful Words / Features (Statistically Significant):


,Coefficient,P-Value,Abs_Impact
sep,-0.029165,2.059098e-07,0.029165
help,-0.021688,7.663088e-02,0.021688
care,-0.020521,5.592956e-02,0.020521
aug,-0.017468,1.459598e-03,0.017468
oct,-0.014854,7.615754e-03,0.014854
is_president,0.014583,3.958481e-10,0.014583
dec,-0.012531,3.039475e-02,0.012531
nov,-0.012449,4.340349e-02,0.012449
mar,-0.011093,4.420404e-02,0.011093
jan,-0.011042,6.259098e-02,0.011042



Effect of being President on Risk:
Statistically significant effect. Coefficient: 0.0146
Top Impactful Words / Features (Statistically Significant):


,Coefficient,P-Value,Abs_Impact
states,-0.094362,0.000435,0.094362
united,0.091918,0.000560,0.091918
doing,-0.071910,0.000809,0.071910
look,-0.050013,0.033951,0.050013
nation,-0.043755,0.017418,0.043755
pm,-0.038832,0.007997,0.038832
care,-0.030638,0.071187,0.030638
national,0.029243,0.091909,0.029243



Effect of being President on Risk:
Not statistically significant at p < 0.05. Coefficient was 0.0019 (p-value: 0.6030)
Top Impactful Words / Features (Statistically Significant):


,Coefficient,P-Value,Abs_Impact
did,0.012866,0.001267,0.012866
total,-0.008798,0.024046,0.008798
china,-0.008518,0.004481,0.008518
states,0.008411,0.093847,0.008411
care,-0.007650,0.016179,0.007650
know,0.007195,0.078823,0.007195
make,-0.006269,0.076836,0.006269



Effect of being President on Risk:
Not statistically significant at p < 0.05. Coefficient was 0.0008 (p-value: 0.2179)


In [86]:
new_sig = significant_factors.copy()
for c in new_sig.columns:
    new_sig[c] = new_sig[c].apply(lambda x: round(x, 3))

with open("outfile.html", "w") as f:
    f.write(new_sig.to_html())

# regression 2: TEB

In [99]:
for fvar in fin_vars:
    for word in res_words[fvar]:
        df_final[word] = df_final['text'].fillna('').apply(lambda x: word in x)
df_final[df_final['is_president'] == 1]

,VIXCLS,Treasury_Yield,Oil_Price,SNP500,VIX_Change,Treasury_Return,Oil_Return,SNP500_Return,text,is_president,...,want,watch,way,whitehouse,win,work,working,world,year,years
date,,,,,,,,,,,,,,,,,,,,,


In [77]:
import statsmodels.formula.api as smf

# add controls for each finvar
for fvar in fin_vars:
    for word in res_words[fvar]:
        df_final[word] = df_final['text'].apply(lambda x: word in x)

# Print the regression summary
# print(model.summary2())



with open("outfile.html", "w") as f:
    f.write(sg.render_html())

# Fit Ordinary Least Squares (OLS) Regression
# model = sm.OLS(Y, X).fit()
models = []

for fvar in fin_vars:
    models.append(smf.ols(formula=f"{fvar} ~ is_president + C(platform) + C(author) + {' + '.join(word for word in res_words[fvar])}", data=df_final).fit())

sg = Stargazer(models)
sg.title("OLS with Presidential and Keyword controls")

sg.custom_columns(
    ["VIX Change", "Treasury Return", "Oil Return", "S&P500 Return"],
    [1, 1, 1, 1]
)

with open("outfile.html", "w") as f:
    f.write(sg.render_html())

In [78]:
assert(df_final['is_president'].sum() != 0)

AssertionError: 

In [46]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

from mizani.breaks import breaks_extended

from plotnine import *

# =====================================================
# SETTINGS
# =====================================================

max_horizon = 12

pretty_names = {
    "VIX_Change": "VIX Change",
    "Treasury_Return": "Treasury Return",
    "Oil_Return": "Oil Return",
    "SNP500_Return": "S&P 500 Return"
}

# Colors from your ggplot example
BLUE = "#005488"
RED  = "#D44000"
GREY40 = "#666666"

# =====================================================
# KEYWORD DUMMIES
# =====================================================

for fvar in fin_vars:
    for word in res_words[fvar]:
        df_final[word] = df_final["text"].apply(
            lambda x: int(word in x)
        )

# =====================================================
# LOCAL PROJECTION FUNCTION
# =====================================================

def run_local_projection(df, yvar, horizons, group_label):

    results = []

    for h in range(horizons + 1):

        df = df.copy()
        future_var = f"{yvar}_lead_{h}"
        df[future_var] = df[yvar].shift(-h)
        df_h = df.dropna(subset=[future_var])

        # --- Drop any regressor that is constant in this subset ---
        cat_controls = [
            f"C({cat})" for cat in ["platform", "author"]
            if df_h[cat].nunique() > 1
        ]

        valid_keywords = [
            w for w in res_words[yvar]
            if w in df_h.columns and df_h[w].nunique() > 1
        ]

        rhs_parts = cat_controls + valid_keywords
        rhs = " + ".join(rhs_parts) if rhs_parts else "1"

        formula = f"{future_var} ~ {rhs}"

        try:
            base_model   = smf.ols(formula=formula, data=df_h).fit()
            nobs         = int(base_model.nobs)
            k            = int(base_model.df_model) + 1
            safe_maxlags = max(1, min(h + 1, nobs - k - 2))

            model = smf.ols(formula=formula, data=df_h).fit(
                cov_type="HAC",
                cov_kwds={"maxlags": safe_maxlags}
            )

            est = model.params["Intercept"]
            se  = model.bse["Intercept"]

        except Exception as e:
            print(f"  Skipping h={h}, group={group_label}, var={yvar}: {e}")
            est, se = np.nan, np.nan

        results.append({
            "h":        h,
            "response": est,
            "lower":    est - 1.645 * se,
            "upper":    est + 1.645 * se,
            "group":    group_label,
            "variable": pretty_names[yvar]
        })

    return pd.DataFrame(results)

# =====================================================
# RUN ALL LPs
# =====================================================

df_pres     = df_final[df_final["is_president"] == 1].copy()
df_non_pres = df_final[df_final["is_president"] == 0].copy()


print("=== df_pres shape ===")
print(df_pres.shape)

print("\n=== is_president value counts in df_final ===")
print(df_final["is_president"].value_counts())

print("\n=== Non-null financial values in df_pres ===")
for fvar in fin_vars:
    print(f"  {fvar}: {df_pres[fvar].notna().sum()} non-null out of {len(df_pres)}")

print("\n=== Sample of df_pres ===")
print(df_pres[["is_president", "text"] + fin_vars].head(10))

all_irfs = []

for fvar in fin_vars:
    all_irfs.append(run_local_projection(df_pres,     fvar, max_horizon, "Presidential"))
    all_irfs.append(run_local_projection(df_non_pres, fvar, max_horizon, "Non-Presidential"))

plot_df = pd.concat(all_irfs)

# =====================================================
# PLOT
# =====================================================

p = (

    ggplot(
        plot_df,
        aes(
            x="h",
            y="response",
            color="group",
            fill="group",
            group="group"
        )
    )

    # ---------------------------------------------
    # Zero line
    # ---------------------------------------------

    + geom_hline(
        yintercept=0,
        linetype="dashed",
        color=GREY40
    )

    # ---------------------------------------------
    # Confidence ribbons
    # ---------------------------------------------

    + geom_ribbon(
        aes(
            ymin="lower",
            ymax="upper"
        ),
        alpha=0.15,
        color=None
    )

    # ---------------------------------------------
    # IRF lines
    # ---------------------------------------------

    + geom_line(
        size=1.4
    )

    # ---------------------------------------------
    # Colors
    # ---------------------------------------------

    + scale_color_manual(
        values={
            "Presidential": BLUE,
            "Non-Presidential": RED
        }
    )

    + scale_fill_manual(
        values={
            "Presidential": BLUE,
            "Non-Presidential": RED
        }
    )

    # ---------------------------------------------
    # Axes
    # ---------------------------------------------

    + scale_x_continuous(
        breaks=range(0, max_horizon + 1)
    )

    + scale_y_continuous(
        breaks=breaks_extended(n=4)
    )

    # ---------------------------------------------
    # Labels
    # ---------------------------------------------

    + labs(
        title="Market Responses to Presidential Communications",
        subtitle="Local projection estimates across financial variables",
        x="Days After Communication",
        y="Estimated Response",
        color=None,
        fill=None,
        caption="90% confidence bands. HAC standard errors."
    )

    # ---------------------------------------------
    # Facets
    # ---------------------------------------------

    + facet_wrap(
        "~variable",
        ncol=2,
        scales="free_y"
    )

    # ---------------------------------------------
    # Theme
    # ---------------------------------------------

    + theme_538()

    + theme(

        # Bigger overall figure
        figure_size=(20, 14),
        dpi=140,

        # Remove legend redundancy
        legend_position="bottom",

        # Facet titles
        strip_text=element_text(
            size=16,
            face="bold"
        ),

        # Plot title
        plot_title=element_text(
            size=24,
            face="bold",
            margin={"b": 12}
        ),

        plot_subtitle=element_text(
            size=16,
            margin={"b": 20}
        ),

        # Axes
        axis_title_x=element_text(
            size=15,
            face="bold",
            vjust=-1
        ),

        axis_title_y=element_text(
            size=15,
            face="bold"
        ),

        axis_text_x=element_text(
            size=13
        ),

        axis_text_y=element_text(
            size=13
        ),

        # Caption
        plot_caption=element_text(
            size=11,
            color=GREY40
        ),

        # More spacing
        subplots_adjust={
            "wspace": 0.25,
            "hspace": 0.30
        }
    )
)

p.draw()

=== df_pres shape ===
(0, 113)

=== is_president value counts in df_final ===
is_president
0    3280
Name: count, dtype: int64

=== Non-null financial values in df_pres ===
  VIX_Change: 0 non-null out of 0
  Treasury_Return: 0 non-null out of 0
  Oil_Return: 0 non-null out of 0
  SNP500_Return: 0 non-null out of 0

=== Sample of df_pres ===
Empty DataFrame
Columns: [is_president, text, VIX_Change, Treasury_Return, Oil_Return, SNP500_Return]
Index: []
  Skipping h=0, group=Presidential, var=VIX_Change: zero-size array to reduction operation maximum which has no identity
  Skipping h=1, group=Presidential, var=VIX_Change: zero-size array to reduction operation maximum which has no identity
  Skipping h=2, group=Presidential, var=VIX_Change: zero-size array to reduction operation maximum which has no identity
  Skipping h=3, group=Presidential, var=VIX_Change: zero-size array to reduction operation maximum which has no identity
  Skipping h=4, group=Presidential, var=VIX_Change: zero-siz

KeyboardInterrupt: 

In [47]:
df_final

,VIXCLS,Treasury_Yield,Oil_Price,SNP500,VIX_Change,Treasury_Return,Oil_Return,SNP500_Return,text,is_president,...,want,watch,way,whitehouse,win,work,working,world,year,years
date,,,,,,,,,,,,,,,,,,,,,
2007-05-07,13.15,4.89,61.48,1509.479980,0.24,-0.01,-0.006625,0.002564,"Barack Obama\n@BarackObama\n·\nMay 7, 2007",0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0000
2007-05-08,13.21,4.90,62.26,1507.719971,0.06,0.01,0.012687,-0.001166,"Barack Obama\n@BarackObama\n·\nMay 8, 2007",0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0000
2007-05-11,12.95,4.87,62.35,1505.849976,-0.65,0.00,0.008084,0.009641,"Barack Obama\n@BarackObama\n·\nMay 10, 2007",0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0000
2007-05-14,13.96,4.85,62.55,1503.150024,1.01,-0.02,0.003208,-0.001793,"Barack Obama\n@BarackObama\n·\nMay 14, 2007",0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0000
2007-05-16,13.50,4.75,62.57,1514.140015,-0.51,-0.08,-0.009341,0.008627,"Barack Obama\n@BarackObama\n·\nMay 16, 2007",0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-22,21.97,4.35,77.68,3822.389893,1.90,0.02,-0.006268,-0.014452,https://www. washingtonexaminer.com/news/wa sh...,0,...,0.0,0.153348,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0000
2022-12-23,20.87,4.34,79.57,3844.820068,-1.10,-0.01,0.024331,0.005868,RT @ realDonaldTrump https:// thefederalist.co...,0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.108421,0.095088,0.00000,0.0889
2022-12-28,22.14,4.46,78.89,3783.219971,0.49,0.00,-0.007048,-0.012021,It has just been learned that the FBI Office t...,0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0000
